# Long-Term Memory — Episodic — LangGraph Agent Tutorial

Episodic memory stores **time-stamped experiences** — what task was attempted and how it went —
so an agent can reference prior work instead of starting from zero. Like semantic memory
(previous notebook), this is implemented with LangGraph's cross-thread `Store`; the difference
is entirely in *how it's written and read*: append-only, ordered by recency, and pruned rather
than updated in place.

In [1]:
# ============ IMPORTS ============
import os
import sys
import sqlite3
import uuid
import json as jsonlib
from datetime import datetime, timezone

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.store.sqlite import SqliteStore
from langgraph.config import get_store

sys.path.append(os.path.abspath("../../.."))
from helpers import get_llm

from dotenv import load_dotenv
load_dotenv()

print("Imports OK")

Imports OK


In [2]:
# ============ LLM + STORE INITIALIZATION ============
llm = get_llm()

DB_PATH = "episodic_memory.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False, isolation_level=None)
episodic_store = SqliteStore(conn)
episodic_store.setup()
print("episodic store ready")

LLM initialized: system.ai.gemma-3-12b (via databricks_gateway)
episodic store ready


## 1. Episodic Tools — Append-Only, Timestamp-Keyed

Unlike semantic memory's stable `key` (so re-saving updates the same row), each episode gets a
**fresh** key — a UUID — because two episodes are never the same fact restated, they're two
distinct events. `store.search` sorted by our own `created_at` field gives us "most recent N,"
the standard episodic read pattern.

In [3]:
# ============ EPISODIC TOOLS ============
_CURRENT_USER: dict = {"id": None}

@tool
def save_episode(task: str, outcome: str, feedback: str = "") -> str:
    """Record a completed task and its outcome as a new episode (never overwrites past episodes)."""
    store = get_store()
    episode_id = str(uuid.uuid4())
    store.put(
        ("episodic", _CURRENT_USER["id"]),
        episode_id,
        {
            "task": task,
            "outcome": outcome,
            "feedback": feedback,
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return "Episode recorded."

@tool
def recall_recent_episodes(limit: int = 3) -> str:
    """Return the most recent episodes for this user."""
    store = get_store()
    items = store.search(("episodic", _CURRENT_USER["id"]))
    items_sorted = sorted(items, key=lambda it: it.value["created_at"], reverse=True)
    recent = [it.value for it in items_sorted[:limit]]
    return jsonlib.dumps(recent, indent=2) if recent else "No past episodes."

episodic_tools = [save_episode, recall_recent_episodes]

## 2. The Agent — Preload Recent Episodes

In [4]:
# ============ MEMORY-AWARE AGENT GRAPH ============
llm_with_tools = llm.bind_tools(episodic_tools)

SYSTEM_TEMPLATE = """You are a helpful assistant with persistent episodic memory.

Recent past interactions with this user:
{episodes}

After you complete a substantive task, call `save_episode` with a short task/outcome summary
so future conversations can reference what was already tried.
"""

def agent_node(state: MessagesState, config: RunnableConfig) -> dict:
    user_id = config["configurable"]["user_id"]
    _CURRENT_USER["id"] = user_id

    store = get_store()
    items = store.search(("episodic", user_id))
    items_sorted = sorted(items, key=lambda it: it.value["created_at"], reverse=True)[:3]
    episodes_block = "\n".join(
        f"- {it.value['task']} -> {it.value['outcome']}" for it in items_sorted
    ) or "(none)"

    system = SystemMessage(SYSTEM_TEMPLATE.format(episodes=episodes_block))
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(episodic_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, ["tools", END])
builder.add_edge("tools", "agent")

episodic_agent = builder.compile(store=episodic_store)
print("Episodic-memory agent compiled.")

Episodic-memory agent compiled.


## 3. A Task Gets Logged as an Episode

In [5]:
# ============ CONVERSATION 1: COMPLETE A TASK ============
USER_ID = "user-alice"
cfg1 = {"configurable": {"user_id": USER_ID, "thread_id": "alice-thread-1"}}

out = episodic_agent.invoke(
    {"messages": [HumanMessage(
        "Help me outline a 3-step plan to migrate a Spark job from Scala to PySpark, "
        "then save this as an episode once you're done."
    )]},
    cfg1,
)
print("AGENT:", out["messages"][-1].content[:400])

AGENT: Okay, great! I've recorded that we outlined a 3-step plan for migrating a Spark job from Scala to PySpark. Let me know what you'd like to do next.


## 4. A Later, New Thread References the Past Episode

In [6]:
# ============ CONVERSATION 2: NEW THREAD, EPISODE RECALLED ============
cfg2 = {"configurable": {"user_id": USER_ID, "thread_id": "alice-thread-2"}}
out2 = episodic_agent.invoke(
    {"messages": [HumanMessage("What did you help me with last time?")]},
    cfg2,
)
print("AGENT:", out2["messages"][-1].content)

AGENT: I helped you with outlining a Spark job migration plan, specifically providing a 3-step plan: Code Translation, Testing and Validation, and Optimization and Refinement. I also outlined a Spark job migration from Scala to PySpark using the same 3-step plan.


## 5. Semantic vs. Episodic — Verified Side by Side

Both notebooks use the exact same `Store` mechanism (`SqliteStore`, namespaced by `user_id`) —
the difference is entirely in the write/read pattern, not the storage primitive:

| | Semantic (previous notebook) | Episodic (this notebook) |
|---|---|---|
| Key | Stable (`"name"`, `"preferred_language"`) | Fresh UUID per event |
| Write behavior | Update-in-place | Append-only |
| Read pattern | Load everything | Load most-recent-N by `created_at` |


## 6. Pruning

Episodic memory grows with every completed task — unlike semantic memory, there's no natural
cap on row count. A retention policy keeps only the most recent `N` episodes per user.

(The padding episodes below are written directly to `episodic_store` rather than through the
`save_episode` tool — `get_store()` only resolves inside an active graph run, so calling a
`@tool` function directly, outside of `ToolNode`, raises. That's worth knowing on its own: see
Gotchas.)

In [ ]:
# ============ PRUNING ============
def prune_old_episodes(user_id: str, keep_last: int = 20) -> int:
    items = episodic_store.search(("episodic", user_id))
    items_sorted = sorted(items, key=lambda it: it.value["created_at"], reverse=True)
    to_delete = items_sorted[keep_last:]
    for it in to_delete:
        episodic_store.delete(("episodic", user_id), it.key)
    return len(to_delete)

# Save a couple more episodes so there's something to prune. `save_episode` is a tool meant to
# run inside a graph (get_store() needs an active run) -- for this demo we write directly to
# the store instead of calling the tool function outside of one.
for i in range(3):
    episodic_store.put(
        ("episodic", USER_ID),
        str(uuid.uuid4()),
        {
            "task": f"Demo task {i}",
            "outcome": f"Demo outcome {i}",
            "feedback": "",
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )

before = len(episodic_store.search(("episodic", USER_ID)))
deleted = prune_old_episodes(USER_ID, keep_last=2)
after = len(episodic_store.search(("episodic", USER_ID)))
print(f"episodes before: {before}, pruned: {deleted}, remaining: {after}")

## Gotchas

- **Append-only means unbounded growth is the default, not an edge case.** Every completed task
  adds a row; without a pruning policy like the one above (or summarization into a compacted
  form), a long-lived user's episode count grows without limit.
- **"Recent N" preloading can miss the relevant episode.** A naive most-recent-3 preload won't
  surface something from months ago — that's when you need `search(..., query=...)` against an
  embedding-indexed store instead of blind recency, or a task-specific keyword filter.
- **Sorting is your responsibility.** `store.search()` doesn't guarantee chronological order —
  this notebook sorts on our own `created_at` field after fetching. Forgetting this step (and
  trusting insertion order) is a common bug once a store is backed by something that doesn't
  preserve write order (e.g. a distributed backend).
- **Episodic entries can embed PII** in `task`/`outcome` text just as easily as semantic facts —
  same access-control caveat applies.
- **Don't duplicate the conversation transcript here.** `save_episode` should record a
  *summary* ("outlined a 3-step migration plan"), not the full exchange — the full exchange is
  short-term memory's job (see `03_Short_Term_Working_Memory_SQLite.ipynb`); copying it into
  episodic memory defeats the purpose of a compact, cheap-to-preload long-term layer.

## Key Takeaways

- Episodic memory in LangGraph reuses the same `Store` primitive as semantic memory, but with an
  append-only write pattern (fresh key per event) and a recency-sorted read pattern.
- Preload the most recent few; prune or summarize the rest — this layer has a distinct growth
  problem semantic memory doesn't.
